In [6]:
!pip3 install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 20.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 w

In [7]:
import folium

In [8]:
import pandas as pd

# URL of the distance matrix CSV
url = 'https://raw.githubusercontent.com/sandrotissi/citibike_vrp/226d6d7b050fafc5805837ba8fefe8a65d2c6996/full_distance_matrix_google.csv'

# Read the CSV into a pandas DataFrame
distance_df = pd.read_csv(url, index_col=0)

# Scale distances by 1000 and convert to integers to avoid OR-Tools truncating small floating-point values to 0
new_distance_matrix = [[int(val * 1000) for val in row] for row in distance_df.values.tolist()]

print(f"Loaded a distance matrix of shape: {len(new_distance_matrix)}x{len(new_distance_matrix[0])}")

Loaded a distance matrix of shape: 111x111


In [9]:
demand_url = 'https://raw.githubusercontent.com/sandrotissi/citibike_vrp/e72b84e218bd17f064948623c0d6d70887f32934/all_archetypes_demand_with_coordinates.csv'
demand_df = pd.read_csv(demand_url)

### Optimization for `demand_001` Archetype

In [10]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

demand_001 = [0] + demand_df['demand_001'].tolist()

def create_data_model_001():
    """Stores the data for the problem."""
    data = {}
    data['distance_matrix'] = new_distance_matrix
    data['demands'] = [int(d) for d in demand_001]
    data['vehicle_capacities'] = [80] * 16
    data['num_vehicles'] = 16
    data['depot'] = 0
    return data

def print_solution_001(data, manager, routing, solution):
    """Prints solution on console and summarizes total distance and vans used."""
    total_distance = 0
    vans_used = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data['demands'][node_index]
            plan_output += f' {node_index} Load({route_load}) -> '
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        node_index = manager.IndexToNode(index)
        plan_output += f'{node_index}\n'
        plan_output += f'Distance of the route: {route_distance}m\n'
        plan_output += f'Load of the route: {route_load}\n'
        print(plan_output)

        if route_distance > 0: # Check if the vehicle was actually used
            total_distance += route_distance
            vans_used += 1
    print(f'Total distance driven by all vehicles: {total_distance}m')
    print(f'Number of vans used: {vans_used}')

def main_001():
    """Solve the CVRP problem."""
    data = create_data_model_001()

    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution_001(data, manager, routing, solution)


if __name__ == '__main__':
    main_001()

Route for vehicle 0:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 1:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 2:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 3:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 4:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 5:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 6:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 7:
 0 Load(0) ->  39 Load(8) ->  84 Load(13) ->  70 Load(17) ->  90 Load(20) ->  89 Load(21) ->  93 Load(23) ->  96 Load(24) ->  81 Load(25) ->  58 Load(27) ->  108 Load(28) ->  82 Load(30) ->  101 Load(31) ->  100 Load(32) ->  99 Load(33) ->  106 Load(34) ->  105 Load(35) ->  104 Load(36) ->  103 Load(37) ->  109 Load(38) ->  102 Load(39) ->  98 Load(41) ->  97 Load(42) ->  107 Load(

### Optimization for `demand_011` Archetype

In [11]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

demand_011 = [0] + demand_df['demand_011'].tolist()

def create_data_model_011():
    """Stores the data for the problem."""
    data = {}
    data['distance_matrix'] = new_distance_matrix
    data['demands'] = [int(d) for d in demand_011]
    data['vehicle_capacities'] = [80] * 16
    data['num_vehicles'] = 16
    data['depot'] = 0
    return data

def print_solution_011(data, manager, routing, solution):
    """Prints solution on console and summarizes total distance and vans used."""
    total_distance = 0
    vans_used = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data['demands'][node_index]
            plan_output += f' {node_index} Load({route_load}) -> '
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        node_index = manager.IndexToNode(index)
        plan_output += f'{node_index}\n'
        plan_output += f'Distance of the route: {route_distance}m\n'
        plan_output += f'Load of the route: {route_load}\n'
        print(plan_output)

        if route_distance > 0:
            total_distance += route_distance
            vans_used += 1
    print(f'Total distance driven by all vehicles: {total_distance}m')
    print(f'Number of vans used: {vans_used}')

def main_011():
    """Solve the CVRP problem."""
    data = create_data_model_011()

    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution_011(data, manager, routing, solution)


if __name__ == '__main__':
    main_011()

Route for vehicle 0:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 1:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 2:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 3:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 4:
 0 Load(0) ->  15 Load(13) ->  17 Load(22) ->  19 Load(31) ->  4 Load(47) ->  30 Load(66) ->  3 Load(80) -> 0
Distance of the route: 38202m
Load of the route: 80

Route for vehicle 5:
 0 Load(0) ->  32 Load(11) ->  27 Load(24) ->  28 Load(40) ->  6 Load(56) ->  10 Load(64) ->  59 Load(70) ->  81 Load(72) ->  108 Load(72) ->  82 Load(75) ->  107 Load(75) ->  41 Load(80) -> 0
Distance of the route: 55104m
Load of the route: 80

Route for vehicle 6:
 0 Load(0) ->  13 Load(14) ->  20 Load(31) ->  11 Load(43) ->  31 Load(52) ->  24 Load(59) ->  25 Load(67) ->  22 Load(67) ->  12 Load(80) -> 0
Distance of the route: 41225m
Load of the ro

### Optimization for `demand_101` Archetype

In [12]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

demand_101 = [0] + demand_df['demand_101'].tolist()

def create_data_model_101():
    """Stores the data for the problem."""
    data = {}
    data['distance_matrix'] = new_distance_matrix
    data['demands'] = [int(d) for d in demand_101]
    data['vehicle_capacities'] = [80] * 16
    data['num_vehicles'] = 16
    data['depot'] = 0
    return data

def print_solution_101(data, manager, routing, solution):
    """Prints solution on console and summarizes total distance and vans used."""
    total_distance = 0
    vans_used = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data['demands'][node_index]
            plan_output += f' {node_index} Load({route_load}) -> '
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        node_index = manager.IndexToNode(index)
        plan_output += f'{node_index}\n'
        plan_output += f'Distance of the route: {route_distance}m\n'
        plan_output += f'Load of the route: {route_load}\n'
        print(plan_output)

        if route_distance > 0:
            total_distance += route_distance
            vans_used += 1
    print(f'Total distance driven by all vehicles: {total_distance}m')
    print(f'Number of vans used: {vans_used}')

def main_101():
    """Solve the CVRP problem."""
    data = create_data_model_101()

    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution_101(data, manager, routing, solution)


if __name__ == '__main__':
    main_101()

Route for vehicle 0:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 1:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 2:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 3:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 4:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 5:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 6:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 7:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 8:
 0 Load(0) ->  80 Load(10) ->  15 Load(18) ->  17 Load(25) ->  10 Load(30) ->  26 Load(39) ->  27 Load(45) ->  32 Load(52) ->  3 Load(62) ->  2 Load(62) ->  1 Load(78) -> 0
Distance of the route: 40879m
Load of the route: 78

Route for vehicle 9:
 0 Load(0) ->  74 Load(3) ->  39 Load(10) ->  84 Loa

### Optimization for `demand_111` Archetype

In [13]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

demand_111 = [0] + demand_df['demand_111'].tolist()

def create_data_model_111():
    """Stores the data for the problem."""
    data = {}
    data['distance_matrix'] = new_distance_matrix
    data['demands'] = [int(d) for d in demand_111]
    data['vehicle_capacities'] = [80] * 16
    data['num_vehicles'] = 16
    data['depot'] = 0
    return data

def print_solution_111(data, manager, routing, solution):
    """Prints solution on console and summarizes total distance and vans used."""
    total_distance = 0
    vans_used = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data['demands'][node_index]
            plan_output += f' {node_index} Load({route_load}) -> '
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        node_index = manager.IndexToNode(index)
        plan_output += f'{node_index}\n'
        plan_output += f'Distance of the route: {route_distance}m\n'
        plan_output += f'Load of the route: {route_load}\n'
        print(plan_output)

        if route_distance > 0:
            total_distance += route_distance
            vans_used += 1
    print(f'Total distance driven by all vehicles: {total_distance}m')
    print(f'Number of vans used: {vans_used}')

def main_111():
    """Solve the CVRP problem."""
    data = create_data_model_111()

    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution_111(data, manager, routing, solution)


if __name__ == '__main__':
    main_111()

Route for vehicle 0:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 1:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 2:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 3:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 4:
 0 Load(0) ->  14 Load(10) ->  18 Load(21) ->  16 Load(29) ->  12 Load(38) ->  13 Load(50) ->  20 Load(64) ->  10 Load(70) ->  9 Load(79) -> 0
Distance of the route: 40163m
Load of the route: 79

Route for vehicle 5:
 0 Load(0) ->  47 Load(6) ->  48 Load(20) ->  69 Load(31) ->  62 Load(39) ->  63 Load(44) ->  68 Load(51) ->  38 Load(69) -> 0
Distance of the route: 37784m
Load of the route: 69

Route for vehicle 6:
 0 Load(0) ->  39 Load(14) ->  84 Load(23) ->  87 Load(25) ->  88 Load(27) ->  66 Load(34) ->  70 Load(40) ->  89 Load(41) ->  93 Load(42) ->  81 Load(44) ->  58 Load(48) ->  92 Load(49) ->  95 Load(50) ->  107 Load(50) -

### Optimization for `demand_100` Archetype

In [14]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

demand_100 = [0] + demand_df['demand_100'].tolist()

def create_data_model_100():
    """Stores the data for the problem."""
    data = {}
    data['distance_matrix'] = new_distance_matrix
    data['demands'] = [int(d) for d in demand_100]
    data['vehicle_capacities'] = [80] * 16
    data['num_vehicles'] = 16
    data['depot'] = 0
    return data

def print_solution_100(data, manager, routing, solution):
    """Prints solution on console and summarizes total distance and vans used."""
    total_distance = 0
    vans_used = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data['demands'][node_index]
            plan_output += f' {node_index} Load({route_load}) -> '
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        node_index = manager.IndexToNode(index)
        plan_output += f'{node_index}\n'
        plan_output += f'Distance of the route: {route_distance}m\n'
        plan_output += f'Load of the route: {route_load}\n'
        print(plan_output)

        if route_distance > 0:
            total_distance += route_distance
            vans_used += 1
    print(f'Total distance driven by all vehicles: {total_distance}m')
    print(f'Number of vans used: {vans_used}')

def main_100():
    """Solve the CVRP problem."""
    data = create_data_model_100()

    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution_100(data, manager, routing, solution)


if __name__ == '__main__':
    main_100()

Route for vehicle 0:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 1:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 2:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 3:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 4:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 5:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 6:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 7:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 8:
 0 Load(0) ->  15 Load(12) ->  17 Load(19) ->  9 Load(26) ->  10 Load(31) ->  12 Load(38) ->  16 Load(42) ->  18 Load(48) ->  14 Load(56) -> 0
Distance of the route: 39618m
Load of the route: 56

Route for vehicle 9:
 0 Load(0) ->  42 Load(6) ->  83 Load(19) ->  55 Load(28) ->  43 Load(36) ->  91 L

### Optimization for `demand_110` Archetype

In [15]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

demand_110 = [0] + demand_df['demand_110'].tolist()

def create_data_model_110():
    """Stores the data for the problem."""
    data = {}
    data['distance_matrix'] = new_distance_matrix
    data['demands'] = [int(d) for d in demand_110]
    data['vehicle_capacities'] = [80] * 16
    data['num_vehicles'] = 16
    data['depot'] = 0
    return data

def print_solution_110(data, manager, routing, solution):
    """Prints solution on console and summarizes total distance and vans used."""
    total_distance = 0
    vans_used = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data['demands'][node_index]
            plan_output += f' {node_index} Load({route_load}) -> '
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        node_index = manager.IndexToNode(index)
        plan_output += f'{node_index}\n'
        plan_output += f'Distance of the route: {route_distance}m\n'
        plan_output += f'Load of the route: {route_load}\n'
        print(plan_output)

        if route_distance > 0:
            total_distance += route_distance
            vans_used += 1
    print(f'Total distance driven by all vehicles: {total_distance}m')
    print(f'Number of vans used: {vans_used}')

def main_110():
    """Solve the CVRP problem."""
    data = create_data_model_110()

    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution_110(data, manager, routing, solution)


if __name__ == '__main__':
    main_110()

Route for vehicle 0:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 1:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 2:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 3:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 4:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 5:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 6:
 0 Load(0) ->  9 Load(8) ->  11 Load(19) ->  20 Load(32) ->  31 Load(38) ->  24 Load(46) ->  25 Load(52) ->  22 Load(52) ->  13 Load(63) ->  12 Load(72) ->  16 Load(80) -> 0
Distance of the route: 41073m
Load of the route: 80

Route for vehicle 7:
 0 Load(0) ->  83 Load(16) ->  55 Load(25) ->  43 Load(35) ->  73 Load(41) ->  78 Load(47) ->  42 Load(55) -> 0
Distance of the route: 41915m
Load of the route: 55

Route for vehicle 8:
 0 Load(0) ->  41 Load(3) ->  59 Lo

### Optimization for `demand_010` Archetype

In [16]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

demand_010 = [0] + demand_df['demand_010'].tolist()

def create_data_model_010():
    """Stores the data for the problem."""
    data = {}
    data['distance_matrix'] = new_distance_matrix
    data['demands'] = [int(d) for d in demand_010]
    data['vehicle_capacities'] = [80] * 16
    data['num_vehicles'] = 16
    data['depot'] = 0
    return data

def print_solution_010(data, manager, routing, solution):
    """Prints solution on console and summarizes total distance and vans used."""
    total_distance = 0
    vans_used = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data['demands'][node_index]
            plan_output += f' {node_index} Load({route_load}) -> '
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        node_index = manager.IndexToNode(index)
        plan_output += f'{node_index}\n'
        plan_output += f'Distance of the route: {route_distance}m\n'
        plan_output += f'Load of the route: {route_load}\n'
        print(plan_output)

        if route_distance > 0:
            total_distance += route_distance
            vans_used += 1
    print(f'Total distance driven by all vehicles: {total_distance}m')
    print(f'Number of vans used: {vans_used}')

def main_010():
    """Solve the CVRP problem."""
    data = create_data_model_010()

    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution_010(data, manager, routing, solution)


if __name__ == '__main__':
    main_010()

Route for vehicle 0:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 1:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 2:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 3:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 4:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 5:
 0 Load(0) ->  13 Load(12) ->  22 Load(12) ->  25 Load(19) ->  24 Load(28) ->  8 Load(37) ->  7 Load(54) ->  6 Load(66) ->  28 Load(80) -> 0
Distance of the route: 41701m
Load of the route: 80

Route for vehicle 6:
 0 Load(0) ->  12 Load(12) ->  10 Load(19) ->  20 Load(34) ->  11 Load(45) ->  27 Load(56) ->  32 Load(66) ->  3 Load(78) -> 0
Distance of the route: 40607m
Load of the route: 78

Route for vehicle 7:
 0 Load(0) ->  45 Load(10) ->  64 Load(16) ->  76 Load(21) ->  53 Load(32) ->  87 Load(33) ->  66 Load(41) ->  65 Load(47) ->  73 Load(54)

### Optimization for `demand_000` Archetype

In [17]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

demand_000 = [0] + demand_df['demand_000'].tolist()

def create_data_model_000():
    """Stores the data for the problem."""
    data = {}
    data['distance_matrix'] = new_distance_matrix
    data['demands'] = [int(d) for d in demand_000]
    data['vehicle_capacities'] = [80] * 16
    data['num_vehicles'] = 16
    data['depot'] = 0
    return data

def print_solution_000(data, manager, routing, solution):
    """Prints solution on console and summarizes total distance and vans used."""
    total_distance = 0
    vans_used = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data['demands'][node_index]
            plan_output += f' {node_index} Load({route_load}) -> '
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        node_index = manager.IndexToNode(index)
        plan_output += f'{node_index}\n'
        plan_output += f'Distance of the route: {route_distance}m\n'
        plan_output += f'Load of the route: {route_load}\n'
        print(plan_output)

        if route_distance > 0:
            total_distance += route_distance
            vans_used += 1
    print(f'Total distance driven by all vehicles: {total_distance}m')
    print(f'Number of vans used: {vans_used}')

def main_000():
    """Solve the CVRP problem."""
    data = create_data_model_000()

    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity')

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution_000(data, manager, routing, solution)


if __name__ == '__main__':
    main_000()

Route for vehicle 0:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 1:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 2:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 3:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 4:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 5:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 6:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 7:
 0 Load(0) -> 0
Distance of the route: 0m
Load of the route: 0

Route for vehicle 8:
 0 Load(0) ->  36 Load(7) ->  35 Load(14) ->  92 Load(16) ->  58 Load(18) ->  81 Load(19) ->  96 Load(20) ->  100 Load(21) ->  99 Load(22) ->  106 Load(23) ->  105 Load(24) ->  104 Load(25) ->  103 Load(26) ->  109 Load(27) ->  102 Load(28) ->  98 Load(30) ->  97 Load(31) ->  101 Load(32) ->  82 Lo

In [18]:
import folium

def get_routes_and_coordinates(data, manager, routing, solution, location_coords, new_waypoints):
    """Extracts routes and corresponding coordinates."""
    all_routes_coords = []

    for vehicle_id in range(data['num_vehicles']):
        route_coords = []
        index = routing.Start(vehicle_id) # This `index` is the depot's internal OR-Tools index

        # 1. Add the depot as the starting point
        depot_node_idx = manager.IndexToNode(index)
        depot_coords = location_coords[depot_node_idx]
        route_coords.append(depot_coords)

        # 2. Add the new waypoints immediately after the depot for visualization (outbound)
        route_coords.extend(new_waypoints)

        # 3. Traverse the OR-Tools solution to get all actual stops
        actual_stops_coords = []
        current_route_index = index
        current_route_index = solution.Value(routing.NextVar(current_route_index))

        while not routing.IsEnd(current_route_index):
            node_index = manager.IndexToNode(current_route_index)
            actual_stops_coords.append(location_coords[node_index])
            current_route_index = solution.Value(routing.NextVar(current_route_index))

        route_coords.extend(actual_stops_coords)

        # 4. Add the new waypoints before returning to the depot (inbound)
        # It's more realistic to traverse them in reverse order
        route_coords.extend(list(reversed(new_waypoints)))

        # 5. Add the final depot coordinate
        # The OR-Tools solution implicitly ends at the depot, so we'll just add the depot coords.
        route_coords.append(depot_coords)

        all_routes_coords.append(route_coords)

    return all_routes_coords

# Re-run main to get the solution object for demand_000
data = create_data_model_000()
manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                       data['num_vehicles'], data['depot'])
routing = pywrapcp.RoutingModel(manager)

# Create and register a transit callback (same as in main)
def distance_callback(from_index, to_index):
    return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]
transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Add Capacity constraint (same as in main)
def demand_callback(from_index):
    return data['demands'][manager.IndexToNode(from_index)]
demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,  # null capacity slack
    data['vehicle_capacities'],  # vehicle maximum capacities
    True,  # start cumul to zero
    'Capacity')

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

solution = routing.SolveWithParameters(search_parameters)

if solution:
    location_coords = {}
    # Get coordinates for all stations from demand_df (node_index 1 corresponds to first row of demand_df)
    for idx, row in demand_df.iterrows():
        # demand_df has 110 rows, corresponding to nodes 1-110
        # idx in demand_df corresponds to node_index = idx + 1
        location_coords[idx + 1] = (row['latitude'], row['longitude'])

    # Add the depot (index 0) with the user-provided coordinates
    depot_lat = 40.646567
    depot_lon = -74.016319
    location_coords[0] = (depot_lat, depot_lon)

    # Define the new waypoints for visualization only (moved outside the function)
    new_waypoints = [
        (40.666659, -73.995811),
        (40.690608, -74.011917),
        (40.704701, -74.016795),
        (40.726184, -74.011653),
        (40.728652, -74.031560)
    ]

    all_routes_coords = get_routes_and_coordinates(data, manager, routing, solution, location_coords, new_waypoints)

    # Create a base map centered around the mean of all coordinates
    map_center = [demand_df['latitude'].mean(), demand_df['longitude'].mean()]
    m = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB Positron')

    # Add depot marker
    # Removed depot marker to prevent confusion based on previous user interaction

    # Define a list of colors for the routes
    colors = ['#FF0000', '#00FF00', '#0000FF', '#FFFF00', '#FF00FF', '#00FFFF', '#800000', '#008000', '#000080', '#808000', '#800080', '#008080', '#C0C0C0', '#808080', '#A52A2A', '#FFA500', '#DDA0DD', '#7FFF00', '#DC143C', '#00CED1']

    # Plot each route
    for i, route_coords in enumerate(all_routes_coords):
        if route_coords:
            folium.PolyLine(route_coords, color=colors[i % len(colors)], weight=2.5, opacity=1).add_to(m)
            # Add markers for each stop in the route (optional, can clutter map)
            for j, coord in enumerate(route_coords):
                # Retrieve depot_coords here for comparison
                depot_coords = location_coords[data['depot']]
                if coord not in new_waypoints and (coord != depot_coords or j == 0):
                    folium.CircleMarker(
                        location=coord,
                        radius=3,
                        color=colors[i % len(colors)],
                        fill=True,
                        fill_color=colors[i % len(colors)],
                        fill_opacity=1,
                        popup=f'Stop {j} on Vehicle {i}'
                    ).add_to(m)

    display(m)